# Study 03: RAG 평가 - RAGAS 이해하기

**목표**: RAG 시스템 평가의 업계 표준인 RAGAS를 이해하고, 4가지 핵심 메트릭을 직접 실습합니다.

**소요 시간**: 약 25분

---

## 1. RAGAS란?

### 이름 풀이

**RAGAS** = **R**etrieval **A**ugmented **G**eneration **A**ssessment **S**ystem

- RAG 시스템의 **"성적표"**를 만들어주는 도구
- 4가지 점수로 RAG 품질을 측정
- **현업 표준** 평가 도구 (논문 기반, 오픈소스)

### 비유로 이해하기

RAG 시스템을 **학생의 오픈북 시험**에 비유하면:

| 단계 | RAG 시스템 | 오픈북 시험 비유 |
|------|-----------|----------------|
| 1. 검색 | 벡터 DB에서 관련 문서 검색 | 교과서에서 관련 페이지 찾기 |
| 2. 생성 | LLM이 검색된 문서로 답변 생성 | 찾은 내용으로 답안 작성 |

RAGAS는 이 과정을 **4가지 관점**에서 평가합니다:

```
질문: "연차휴가는 며칠인가요?"

[검색된 문서]
- "1년차 11일, 2년차 15일, 3년차 이상 20일의 연차가 부여됩니다."
- "휴가 사용 시 1주일 전 신청 필요"

[생성된 답변]
"1년차는 11일, 2년차는 15일, 3년차 이상은 20일입니다."

→ RAGAS가 4가지 점수 부여!
```

## 2. 왜 RAGAS를 쓰나요?

### 기존 방식의 문제점

```python
# 방법 1: 키워드 매칭 (너무 단순함)
keywords = ["11일", "15일", "연차"]
score = sum(1 for k in keywords if k in answer) / len(keywords)

# 문제: "연차는 11일부터 시작" vs "1년차 직원에게 11일 부여"
# → 같은 의미인데 키워드 위치/문맥 무시
```

```python
# 방법 2: 직접 LLM 평가 (비표준)
prompt = f"이 답변이 좋은지 0-10점으로 평가해줘: {answer}"

# 문제: 평가 기준이 모호, 재현성 없음, 다른 프로젝트와 비교 불가
```

### RAGAS의 장점

| 항목 | 기존 방식 | RAGAS |
|------|----------|-------|
| 표준화 | ❌ 각자 다른 방식 | ✅ 업계 표준 메트릭 |
| 재현성 | ❌ 프롬프트마다 다름 | ✅ 동일 조건 = 동일 결과 |
| 비교 가능 | ❌ 프로젝트 간 비교 불가 | ✅ 논문/타사와 비교 가능 |
| 관점 | ❌ 단일 점수 | ✅ 4가지 세분화된 관점 |

## 3. RAGAS 4가지 메트릭

### 한눈에 보기

| 메트릭 | 질문 | 비유 | 범위 |
|--------|------|------|------|
| **Faithfulness** | 답변이 검색된 문서 내용만 사용했나? | 교과서에 있는 내용만 답했나? | 0~1 |
| **Answer Relevancy** | 답변이 질문과 관련있나? | 질문에 맞는 답을 했나? | 0~1 |
| **Context Precision** | 검색된 문서가 질문과 관련있나? | 관련 자료를 잘 찾았나? | 0~1 |
| **Context Recall** | 필요한 정보를 다 가져왔나? | 빠뜨린 자료 없나? | 0~1 |

---

### 3.1 Faithfulness (충실도)

**질문**: "답변이 검색된 문서 내용만 사용했나?"

```
[검색된 문서]
"1년차 11일, 2년차 15일의 연차가 부여됩니다."

[좋은 답변] Faithfulness = 1.0
"1년차는 11일, 2년차는 15일입니다."
→ 문서 내용만 사용 ✓

[나쁜 답변] Faithfulness = 0.0
"1년차는 11일이고, 보통 대기업은 15일을 줍니다."
→ "대기업은 15일" = 문서에 없는 내용 (환각!)
```

**중요**: Faithfulness가 낮으면 **환각(Hallucination)** 발생!

---

### 3.2 Answer Relevancy (답변 관련성)

**질문**: "답변이 질문에 맞는 내용인가?"

```
[질문] "연차휴가는 며칠인가요?"

[좋은 답변] Answer Relevancy = 1.0
"1년차 11일, 2년차 15일입니다."
→ 질문에 정확히 답함 ✓

[나쁜 답변] Answer Relevancy = 0.3
"연차휴가는 1주일 전에 신청해야 합니다. 승인은 팀장이 합니다."
→ 연차 관련이긴 하지만, "며칠?"에 답하지 않음
```

---

### 3.3 Context Precision (문맥 정밀도)

**질문**: "검색된 문서들이 질문과 관련있나?"

```
[질문] "연차휴가는 며칠인가요?"

[좋은 검색] Context Precision = 1.0
- 문서1: "1년차 11일, 2년차 15일..." ← 관련 ✓
- 문서2: "연차 사용 시 신청 절차..." ← 관련 ✓

[나쁜 검색] Context Precision = 0.33
- 문서1: "1년차 11일, 2년차 15일..." ← 관련 ✓
- 문서2: "회사 식당 운영 시간..." ← 무관 ✗
- 문서3: "주차장 이용 안내..." ← 무관 ✗
```

---

### 3.4 Context Recall (문맥 재현율)

**질문**: "답변에 필요한 정보를 모두 검색했나?"

```
[질문] "연차휴가는 며칠인가요?"
[정답(Ground Truth)] "1년차 11일, 2년차 15일, 3년차 이상 20일"

[좋은 검색] Context Recall = 1.0
- 검색된 문서에 1년차, 2년차, 3년차 정보 모두 포함 ✓

[나쁜 검색] Context Recall = 0.5
- 검색된 문서에 1년차, 2년차만 있고 3년차 정보 없음 ✗
```

**참고**: Context Recall은 `ground_truth`(정답)가 필요합니다.

---

## 4. 환경 설정

RAGAS를 실습하기 위한 환경을 설정합니다.

In [1]:
# 패키지 설치 (처음 한 번만 실행)
!pip install ragas -q

In [2]:
# 임포트 및 경로 설정
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# 프로젝트 루트 설정
PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

# .env 파일 로드
load_dotenv(PROJECT_ROOT / ".env")

# Windows 환경 호환성
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(f"프로젝트 루트: {PROJECT_ROOT}")

# RAGAS 임포트 (최신 API)
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.llms import llm_factory
from langchain_openai import OpenAIEmbeddings
from ragas.metrics._faithfulness import Faithfulness
from ragas.metrics._answer_relevance import AnswerRelevancy
from ragas.metrics._context_precision import ContextPrecision
from ragas.metrics._context_recall import ContextRecall
from ragas.run_config import RunConfig

print("임포트 완료!")

프로젝트 루트: C:\workspace\enterprise-hr-agent


C:\Users\82109\miniconda3\envs\hr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


임포트 완료!


---

## 5. RAGAS 데이터셋 형식

### SingleTurnSample 이해하기

RAGAS 평가를 위해서는 데이터를 특정 형식으로 준비해야 합니다.

```python
from ragas import SingleTurnSample

sample = SingleTurnSample(
    user_input="질문",                    # 사용자 질문
    retrieved_contexts=["문서1", "문서2"],  # 검색된 문서들 (리스트)
    response="생성된 답변",                # LLM이 생성한 답변
    reference="정답"                      # Ground Truth (선택사항)
)
```

### 필드별 용도

| 필드 | 설명 | 필요한 메트릭 |
|------|------|-------------|
| `user_input` | 사용자 질문 | 모든 메트릭 |
| `retrieved_contexts` | 검색된 문서 리스트 | Faithfulness, Context Precision/Recall |
| `response` | LLM 생성 답변 | Faithfulness, Answer Relevancy |
| `reference` | 정답 (Ground Truth) | Context Recall |

In [3]:
# 예제: SingleTurnSample 생성하기

# 1. 단일 샘플 생성
sample1 = SingleTurnSample(
    user_input="연차휴가는 며칠인가요?",
    retrieved_contexts=[
        "1년차 11일, 2년차 15일, 3년차 이상 20일의 연차가 부여됩니다.",
        "연차휴가 사용 시 1주일 전 신청이 필요합니다."
    ],
    response="1년차는 11일, 2년차는 15일, 3년차 이상은 20일입니다.",
    reference="1년차 11일, 2년차 15일, 3년차 이상 20일"
)

print("=== 샘플 1 (좋은 RAG) ===")
print(f"질문: {sample1.user_input}")
print(f"검색된 문서 수: {len(sample1.retrieved_contexts)}")
print(f"생성된 답변: {sample1.response}")
print(f"정답: {sample1.reference}")

# 2. 나쁜 예제 (환각 포함)
sample2 = SingleTurnSample(
    user_input="연차휴가는 며칠인가요?",
    retrieved_contexts=[
        "1년차 11일, 2년차 15일의 연차가 부여됩니다."
    ],
    response="1년차는 11일이고, 대기업은 보통 20일을 줍니다.",  # 환각!
    reference="1년차 11일, 2년차 15일, 3년차 이상 20일"
)

print("\n=== 샘플 2 (나쁜 RAG - 환각 포함) ===")
print(f"질문: {sample2.user_input}")
print(f"생성된 답변: {sample2.response}")
print("→ '대기업은 보통 20일' = 문서에 없는 내용 (환각!)")

# 3. EvaluationDataset 생성
dataset = EvaluationDataset(samples=[sample1, sample2])
print(f"\n데이터셋 생성 완료: {len(dataset)} 샘플")

=== 샘플 1 (좋은 RAG) ===
질문: 연차휴가는 며칠인가요?
검색된 문서 수: 2
생성된 답변: 1년차는 11일, 2년차는 15일, 3년차 이상은 20일입니다.
정답: 1년차 11일, 2년차 15일, 3년차 이상 20일

=== 샘플 2 (나쁜 RAG - 환각 포함) ===
질문: 연차휴가는 며칠인가요?
생성된 답변: 1년차는 11일이고, 대기업은 보통 20일을 줍니다.
→ '대기업은 보통 20일' = 문서에 없는 내용 (환각!)

데이터셋 생성 완료: 2 샘플


---

## 6. RAGAS 평가 실행

이제 위에서 만든 데이터셋으로 실제 RAGAS 평가를 실행합니다.

### 평가 과정

1. **LLM 준비**: RAGAS는 내부적으로 LLM을 사용해 평가 (평가자 LLM)
2. **메트릭 선택**: 4가지 중 원하는 메트릭 선택
3. **평가 실행**: `evaluate()` 함수 호출
4. **결과 확인**: 각 샘플별, 전체 평균 점수 확인

In [4]:
# 1. 평가자 LLM 및 Embeddings 설정 (OpenAI gpt-4o-mini 사용)
from openai import OpenAI

# OpenAI 클라이언트 설정
openai_client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

# LLM 설정 (gpt-4o-mini - 저렴하고 신뢰할 수 있는 평가자)
evaluator_llm = llm_factory(
    model="gpt-4o-mini",
    provider="openai",
    client=openai_client
)

# Embeddings 설정 (AnswerRelevancy에 필요)
evaluator_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)

print("=== RAGAS 평가자 설정 ===")
print(f"평가자 LLM: gpt-4o-mini (OpenAI)")
print(f"Embeddings: text-embedding-3-small")
print(f"※ 평가 대상은 Cell 15의 RAGAgent (qwen3:8b)입니다")

# 2. 4가지 메트릭 정의
metrics = [
    Faithfulness(llm=evaluator_llm),                                    # 충실도
    AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),  # 답변 관련성
    ContextPrecision(llm=evaluator_llm),                                # 문맥 정밀도
    ContextRecall(llm=evaluator_llm),                                   # 문맥 재현율
]

print("\n평가 메트릭:")
for m in metrics:
    print(f"  - {m.name}")

# 3. RunConfig 설정 (OpenAI용 - 병렬 처리 가능)
run_config = RunConfig(
    max_workers=4,   # OpenAI는 병렬 처리 가능
    timeout=120      # 타임아웃 2분
)

print(f"\nRunConfig: max_workers={run_config.max_workers}, timeout={run_config.timeout}s")

=== RAGAS 평가자 설정 ===
평가자 LLM: gpt-4o-mini (OpenAI)
Embeddings: text-embedding-3-small
※ 평가 대상은 Cell 15의 RAGAgent (qwen3:8b)입니다

평가 메트릭:
  - faithfulness
  - answer_relevancy
  - context_precision
  - context_recall

RunConfig: max_workers=4, timeout=120s


In [5]:
%%time
# 4. 평가 실행 (약 2-3분 소요)
print("RAGAS 평가 실행 중...\n")

result = evaluate(
    dataset=dataset,
    metrics=metrics,
    run_config=run_config
)

print("평가 완료!")

RAGAS 평가 실행 중...



Evaluating: 100%|███████████████████████████████████████████████████████████████████| 8/8 [00:42<00:00,  5.31s/it]


평가 완료!
CPU times: total: 11.5 s
Wall time: 47.4 s


In [6]:
# 5. 결과 확인 및 해석

def get_score(result, metric_name):
    """결과에서 점수 추출 (리스트면 평균)"""
    value = result[metric_name]
    if isinstance(value, list):
        valid = [v for v in value if v is not None and v == v]  # NaN 제외
        return valid  # 개별 점수 리스트 반환
    return value

print("=" * 60)
print("RAGAS 평가 결과")
print("=" * 60)

# 전체 평균
print("\n[전체 평균]")
for metric in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    scores = get_score(result, metric)
    if isinstance(scores, list):
        avg = sum(scores) / len(scores) if scores else 0
        print(f"  {metric}: {avg:.3f}")
    else:
        print(f"  {metric}: {scores:.3f}")

# 샘플별 비교
print("\n[샘플별 비교]")
print("-" * 60)

faithfulness_scores = get_score(result, "faithfulness")
relevancy_scores = get_score(result, "answer_relevancy")

print("샘플 1 (좋은 RAG):")
print(f"  - 질문: {sample1.user_input}")
print(f"  - Faithfulness: {faithfulness_scores[0]:.3f}" if isinstance(faithfulness_scores, list) and len(faithfulness_scores) > 0 else "  - Faithfulness: N/A")
print(f"  - Answer Relevancy: {relevancy_scores[0]:.3f}" if isinstance(relevancy_scores, list) and len(relevancy_scores) > 0 else "  - Answer Relevancy: N/A")

print("\n샘플 2 (나쁜 RAG - 환각 포함):")
print(f"  - 질문: {sample2.user_input}")
print(f"  - Faithfulness: {faithfulness_scores[1]:.3f}" if isinstance(faithfulness_scores, list) and len(faithfulness_scores) > 1 else "  - Faithfulness: N/A")
print(f"  - Answer Relevancy: {relevancy_scores[1]:.3f}" if isinstance(relevancy_scores, list) and len(relevancy_scores) > 1 else "  - Answer Relevancy: N/A")
print("  → 환각이 포함되어 Faithfulness가 낮아야 함!")

RAGAS 평가 결과

[전체 평균]
  faithfulness: 0.750
  answer_relevancy: 0.367
  context_precision: 1.000
  context_recall: 0.500

[샘플별 비교]
------------------------------------------------------------
샘플 1 (좋은 RAG):
  - 질문: 연차휴가는 며칠인가요?
  - Faithfulness: 1.000
  - Answer Relevancy: 0.299

샘플 2 (나쁜 RAG - 환각 포함):
  - 질문: 연차휴가는 며칠인가요?
  - Faithfulness: 0.500
  - Answer Relevancy: 0.435
  → 환각이 포함되어 Faithfulness가 낮아야 함!


---

## 7. 실제 프로젝트 적용

이제 우리 프로젝트의 `rag_test.json` 데이터로 RAG Agent를 평가해봅시다.

In [7]:
# 1. 테스트 데이터 로드
test_file = PROJECT_ROOT / "data/finetuning/rag_test.json"

with open(test_file, "r", encoding="utf-8") as f:
    test_cases = json.load(f)

print(f"테스트 케이스: {len(test_cases)}개")
print("\n예시:")
for i, case in enumerate(test_cases[:3]):
    print(f"  {i+1}. {case['question']}")
    print(f"     정답: {case['ground_truth'][:50]}...")

테스트 케이스: 10개

예시:
  1. 병가는 어떻게 사용하나요?
     정답: 병가는 질병 또는 부상으로 인해 근무가 어려운 경우 사용하는 휴가입니다....
  2. 육아휴직은 얼마 동안 쓸 수 있어?
     정답: 육아휴직은 만 8세 이하 자녀를 양육하기 위해 일정 기간 근무를 중단하는 것입니다....
  3. 정직 징계가 뭐야?
     정답: 정직은 일정 기간 출근을 정지하고 급여를 지급하지 않는 징계입니다....


In [11]:
%%time
# 2. RAG Agent로 응답 생성 및 평가
from core.agents.rag_agent import RAGAgent

# RAG Agent 초기화
rag_agent = RAGAgent(
    model=os.environ.get("OLLAMA_MODEL", "qwen3:8b"),
    provider="ollama",
    base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
)

print("RAG Agent 초기화 완료!")
print(f"모델: {rag_agent.model}")

# RAGAS 샘플 생성
samples = []
print("\n응답 생성 중...")

for i, case in enumerate(test_cases):
    question = case["question"]
    ground_truth = case["ground_truth"]
    
    # RAG Agent 호출
    response = rag_agent.query(question)
    answer = response["answer"]
    
    # retriever에서 직접 전체 문서 가져오기 (RAGAS 평가용)
    # response["metadata"]["source_docs"]는 200자로 잘려있어서 RAGAS 평가에 부적합
    docs = rag_agent.retriever.invoke(question)
    retrieved_contexts = [doc.page_content for doc in docs]
    
    # 디버깅 출력 (첫 번째 샘플만)
    if i == 0:
        print(f"    → contexts 수: {len(retrieved_contexts)}")
        if retrieved_contexts:
            print(f"    → 첫 context 길이: {len(retrieved_contexts[0])}자")
    
    # RAGAS 샘플 생성
    sample = SingleTurnSample(
        user_input=question,
        retrieved_contexts=retrieved_contexts,
        response=answer,
        reference=ground_truth
    )
    samples.append(sample)
    
    print(f"  [{i+1}/{len(test_cases)}] {question[:30]}...")

# 평가 데이터셋 생성
eval_dataset = EvaluationDataset(samples=samples)
print(f"\n평가 데이터셋: {len(eval_dataset)} 샘플")

RAG Agent 초기화 완료!
모델: qwen3-hr

응답 생성 중...
    → contexts 수: 5
    → 첫 context 길이: 280자
  [1/10] 병가는 어떻게 사용하나요?...
  [2/10] 육아휴직은 얼마 동안 쓸 수 있어?...
  [3/10] 정직 징계가 뭐야?...
  [4/10] 재택근무는 어떻게 하는 거야?...
  [5/10] 연장근로 수당은 어떻게 계산해?...
  [6/10] 경조사휴가 종류 알려줘...
  [7/10] 수습기간이 뭐야?...
  [8/10] 복지포인트는 어디에 쓸 수 있어?...
  [9/10] 안식휴가는 언제 받을 수 있어?...
  [10/10] 회사 VPN은 뭐야?...

평가 데이터셋: 10 샘플
CPU times: total: 1.11 s
Wall time: 39.3 s


In [12]:
%%time
# 3. RAGAS 평가 실행
print("RAGAS 평가 실행 중... (약 5-10분 소요)\n")

eval_result = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    run_config=run_config
)

print("평가 완료!")

RAGAS 평가 실행 중... (약 5-10분 소요)



Evaluating: 100%|█████████████████████████████████████████████████████████████████| 40/40 [06:00<00:00,  9.02s/it]


평가 완료!
CPU times: total: 1min 4s
Wall time: 6min 5s


---

## 8. 결과 해석 가이드

### 점수별 해석

| 점수 범위 | 의미 | 조치 |
|----------|------|------|
| **0.8 ~ 1.0** | 우수 | 현재 설정 유지 |
| **0.6 ~ 0.8** | 양호 | 개선 여지 있음 |
| **0.4 ~ 0.6** | 보통 | 개선 필요 |
| **0.0 ~ 0.4** | 미흡 | 즉시 개선 필요 |

### 메트릭별 개선 방법

| 메트릭 | 낮을 때 원인 | 개선 방법 |
|--------|-------------|----------|
| **Faithfulness** | LLM이 문서 외 내용 생성 | 프롬프트에 "문서 내용만 사용" 강조 |
| **Answer Relevancy** | 질문과 동떨어진 답변 | 프롬프트 개선, 모델 교체 |
| **Context Precision** | 무관한 문서 검색 | chunk_size 조정, 임베딩 모델 변경 |
| **Context Recall** | 필요한 정보 누락 | top_k 증가, chunk_overlap 증가 |

In [13]:
# 결과 출력 및 해석
print("=" * 60)
print("RAG Agent RAGAS 평가 결과")
print("=" * 60)

print("\n[전체 평균]")
for metric in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
    scores = get_score(eval_result, metric)
    if isinstance(scores, list):
        avg = sum(scores) / len(scores) if scores else 0
        # 해석 추가
        if avg >= 0.8:
            status = "✅ 우수"
        elif avg >= 0.6:
            status = "🔶 양호"
        elif avg >= 0.4:
            status = "⚠️ 보통"
        else:
            status = "❌ 미흡"
        print(f"  {metric}: {avg:.3f} {status}")
    else:
        print(f"  {metric}: {scores:.3f}")

print("\n" + "=" * 60)
print("결과 해석")
print("=" * 60)
print("""
• Faithfulness (충실도): 답변이 검색된 문서 내용만 사용했는지
  → 낮으면 환각(Hallucination) 발생 가능성

• Answer Relevancy (답변 관련성): 답변이 질문에 맞는지
  → 낮으면 프롬프트 또는 모델 개선 필요

• Context Precision (문맥 정밀도): 검색된 문서가 관련 있는지
  → 낮으면 청킹/임베딩 설정 조정 필요

• Context Recall (문맥 재현율): 필요한 정보를 다 검색했는지
  → 낮으면 top_k 증가 또는 chunk_overlap 조정
""")

RAG Agent RAGAS 평가 결과

[전체 평균]
  faithfulness: 0.000 ❌ 미흡
  answer_relevancy: 0.107 ❌ 미흡
  context_precision: 0.789 🔶 양호
  context_recall: 1.000 ✅ 우수

결과 해석

• Faithfulness (충실도): 답변이 검색된 문서 내용만 사용했는지
  → 낮으면 환각(Hallucination) 발생 가능성

• Answer Relevancy (답변 관련성): 답변이 질문에 맞는지
  → 낮으면 프롬프트 또는 모델 개선 필요

• Context Precision (문맥 정밀도): 검색된 문서가 관련 있는지
  → 낮으면 청킹/임베딩 설정 조정 필요

• Context Recall (문맥 재현율): 필요한 정보를 다 검색했는지
  → 낮으면 top_k 증가 또는 chunk_overlap 조정



---

## 9. 핵심 정리 및 면접 답변

### 학습 완료 체크리스트

- [ ] RAGAS가 무엇인지 설명할 수 있다
- [ ] 4가지 메트릭의 의미를 이해한다
- [ ] SingleTurnSample 형식을 알고 있다
- [ ] 점수별 해석 방법을 안다
- [ ] 메트릭별 개선 방법을 안다

### 면접 답변 요점

#### Q: RAG 시스템을 어떻게 평가했나요?

> "**RAGAS 프레임워크**로 4가지 메트릭을 측정했습니다.
> 
> 1. **Faithfulness**: 환각 여부 확인 (답변이 검색 문서만 사용하는지)
> 2. **Answer Relevancy**: 답변이 질문에 적합한지
> 3. **Context Precision**: 검색 품질 (관련 문서 비율)
> 4. **Context Recall**: 정보 누락 여부
> 
> 이 메트릭들로 chunk_size, overlap, top_k 파라미터를 최적화했습니다."

#### Q: 왜 RAGAS를 선택했나요?

> "**업계 표준**이기 때문입니다.
> 
> - 논문 기반으로 검증된 메트릭
> - 다른 프로젝트/논문과 비교 가능
> - 4가지 관점으로 세분화된 평가 가능
> - 직접 만든 평가보다 재현성과 객관성이 높음"

#### Q: Faithfulness가 낮으면 어떻게 개선하나요?

> "**환각(Hallucination)** 문제이므로:
> 
> 1. 프롬프트에 '검색된 문서 내용만 사용하라' 명시
> 2. Temperature를 0으로 설정
> 3. 문서에서 직접 인용하도록 유도
> 4. 더 신뢰할 수 있는 LLM 모델로 교체"

---

## 10. 다음 단계

**step_03_rag_evaluation.ipynb**: 실제 구현

- Base 모델 vs Fine-tuned 모델 비교
- 전체 테스트셋 평가
- 파라미터별 RAGAS 점수 비교
- 결과 저장 및 시각화

---

**수고하셨습니다!** 🎉

이제 RAGAS가 무엇인지, 4가지 메트릭의 의미가 무엇인지 이해했습니다.